# scPolASeq — APA Analysis

Visualises APA features, coverage tracks and differential poly-A site usage  
across KMeans clusters from the pbmc1k test run.

In [ ]:
from pathlib import Path

RESULTS_DIR  = Path("/scratch/results/scPolASeq/pbmc1k")
LABELS_DIR   = RESULTS_DIR / "labels"
COVERAGE_DIR = RESULTS_DIR / "coverage"
APA_CALLS    = RESULTS_DIR / "apa_calls"
APA_FEATURES = RESULTS_DIR / "apa_features" / "apa_features.tsv"
PAS_REF      = RESULTS_DIR / "pas_reference" / "pas_reference.tsv"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1  Cluster overview

In [ ]:
anno = pd.read_csv(LABELS_DIR / "cell_annotations.tsv", sep="\t")
group_map = pd.read_csv(LABELS_DIR / "group_map.tsv", sep="\t")

print("Cell annotations:", len(anno), "cells")
print(anno["cluster_id"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(9, 4))
counts = anno["cluster_id"].value_counts().sort_index()
counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white", width=0.7)
ax.set_xlabel("Cluster")
ax.set_ylabel("# cells")
ax.set_title("Cells per cluster")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 2  APA feature space

`apa_features.tsv` has one row per (site × group_level × group_id) combination.

In [ ]:
print("Loading APA features…", APA_FEATURES)
feat = pd.read_csv(APA_FEATURES, sep="\t")
print(f"{len(feat):,} rows  ×  {len(feat.columns)} columns")
feat.head(3)

In [ ]:
# Separate by group level
cluster_feat = feat[feat["group_level"] == "cluster"].copy()
ct_feat      = feat[feat["group_level"] == "cell_type"].copy()
print("Cluster-level features:", len(cluster_feat))
print("Cell-type-level features:", len(ct_feat))

# Sites with any non-zero coverage in cluster-level features
covered = cluster_feat[cluster_feat["coverage_at_site"] > 0]
print(f"Sites with coverage > 0: {covered['site_id'].nunique():,}")

In [ ]:
if len(covered) == 0:
    print("⚠  No covered sites yet — coverage tracks are empty.")
    print("   Rerun the pipeline after the bam_to_bedgraph.py fix to populate bedGraphs.")
else:
    # Coverage distribution per group
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # log1p of coverage at site
    for gid, grp in covered.groupby("group_id"):
        axes[0].hist(np.log1p(grp["coverage_at_site"]), bins=40,
                     alpha=0.5, label=gid, density=True)
    axes[0].set_xlabel("log1p(coverage at site)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Coverage distribution")
    axes[0].legend(fontsize=7)

    # Read-end density distribution
    for gid, grp in covered.groupby("group_id"):
        axes[1].hist(grp["read_end_density"], bins=40,
                     alpha=0.5, label=gid, density=True)
    axes[1].set_xlabel("Read-end density")
    axes[1].set_ylabel("Density")
    axes[1].set_title("3' read-end density")
    axes[1].legend(fontsize=7)

    plt.tight_layout()
    plt.show()

## 3  Proximal/distal ratios per cluster

In [ ]:
if len(covered) > 0:
    pdr = (
        covered.groupby("group_id")["proximal_distal_ratio"]
        .describe()
        .round(3)
    )
    display(pdr)

    fig, ax = plt.subplots(figsize=(10, 4))
    groups = sorted(covered["group_id"].unique())
    data   = [covered.loc[covered["group_id"] == g, "proximal_distal_ratio"].dropna() for g in groups]
    bp = ax.boxplot(data, labels=groups, patch_artist=True, notch=False)
    for patch in bp["boxes"]:
        patch.set_facecolor("steelblue")
        patch.set_alpha(0.6)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Proximal / Distal ratio")
    ax.set_title("Poly-A site proximal/distal usage per cluster")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No coverage yet.")

## 4  APA events (differential poly-A site usage)

In [ ]:
apa_events_path = APA_CALLS / "apa_events.tsv"
apa_events = pd.read_csv(apa_events_path, sep="\t")
print(f"APA events: {len(apa_events)} rows")

if len(apa_events) == 0:
    print("⚠  No APA events called yet — this will populate after the coverage fix rerun.")
else:
    print(apa_events.head())

In [ ]:
scored_path = APA_CALLS / "scored_apa_events.tsv"
scored = pd.read_csv(scored_path, sep="\t")
print(f"Scored APA events: {len(scored)} rows")

if len(scored) > 0:
    print(scored.head())
    print("\nColumns:", scored.columns.tolist())
    
    if "pdui_delta" in scored.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(scored["pdui_delta"].dropna(), bins=50,
                     color="steelblue", edgecolor="none", alpha=0.8)
        axes[0].axvline(0, color="red", ls="--")
        axes[0].set_xlabel("ΔPDUI")
        axes[0].set_title("ΔPDUI distribution")
        axes[0].set_ylabel("Events")

        if "apa_score" in scored.columns:
            axes[1].scatter(scored["pdui_delta"], scored["apa_score"],
                            alpha=0.3, s=3, c="steelblue")
            axes[1].set_xlabel("ΔPDUI")
            axes[1].set_ylabel("APA score")
            axes[1].set_title("APA score vs ΔPDUI")
        plt.tight_layout()
        plt.show()
else:
    print("⚠  No scored APA events yet.")

## 5  PAS reference summary

In [ ]:
pas_files = list((RESULTS_DIR / "pas_reference").glob("*.tsv"))
print("PAS reference files:", [f.name for f in pas_files])

if pas_files:
    pas = pd.read_csv(pas_files[0], sep="\t")
    print(f"{len(pas):,} sites  ×  {len(pas.columns)} columns")
    print(pas.head(3))
    print("\nSource distribution:")
    if "source" in pas.columns:
        print(pas["source"].value_counts())
    if "strand" in pas.columns:
        fig, ax = plt.subplots(figsize=(8, 3))
        chrom_counts = pas["chrom"].value_counts().head(25)
        chrom_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
        ax.set_title("PAS sites per chromosome (top 25)")
        ax.set_xlabel("Chromosome")
        ax.set_ylabel("# sites")
        plt.xticks(rotation=45, ha="right", fontsize=8)
        plt.tight_layout()
        plt.show()

## 6  Coverage bedGraph inspection

Quick check to confirm coverage tracks are populated after the bedtools→pysam fix.

In [ ]:
bg_stats = []
for level in ["cluster", "cell_type"]:
    level_dir = COVERAGE_DIR / level
    if level_dir.exists():
        for bg in sorted(level_dir.glob("*.fwd.bedGraph")):
            size = bg.stat().st_size
            # Count intervals
            try:
                n_lines = sum(1 for _ in open(bg))
            except Exception:
                n_lines = 0
            bg_stats.append({"level": level, "group": bg.stem.replace(".fwd", ""),
                              "strand": "fwd", "size_kb": round(size/1024, 1),
                              "n_intervals": n_lines})

bg_df = pd.DataFrame(bg_stats)
print(bg_df.to_string(index=False))

if bg_df["n_intervals"].sum() == 0:
    print("\n⚠  All bedGraphs are empty — rerun pipeline after coverage fix.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    cluster_bg = bg_df[bg_df["level"] == "cluster"]
    ax.bar(cluster_bg["group"], cluster_bg["n_intervals"], color="steelblue", edgecolor="white")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("# bedGraph intervals (fwd strand)")
    ax.set_title("Coverage track size per cluster")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()